# Workout Tracking, (With Google Sheets)

In [ ]:
from pathlib import Path
import requests
from dotenv import load_dotenv
import os

In [ ]:
env_path = str(Path.cwd().parent)+'/.env'
load_dotenv(env_path)

NUTRITION_KEY = os.getenv('Nutrition_APIKey')
APP_ID = os.getenv('App_Nutrition_ID')

header={
    'x-app-id': APP_ID,
    'x-app-key': NUTRITION_KEY,
}

In [ ]:
Base_endpoint = 'https://app.100daysofpython.dev'
Post_endpoint = '/v1/nutrition/natural/exercise'
#Check the server status and database connectivity
Full_get_endpoint = Base_endpoint+'/healthz'
#to send a petition
Full_post_endpoint = Base_endpoint+Post_endpoint

# Test for post with the API

## PARAMETERS
{

  "query": "ran 3 miles",            // **Required**: Exercise description

  "weight_kg": 70,                  // Optional: Weight in kg (1-500)

  "height_cm": 175,                 // Optional: Height in cm (1-300)

  "age": 30,                        // Optional: Age (1-150)

  "gender": "male"                  // Optional: "male" or "female"
  
}

In [ ]:
#We only will fill for now the required one 
parameters = {
    "query": "ran for 30 min"
}

In [ ]:

response = requests.post(url=Full_post_endpoint, headers=header, json=parameters)
response.raise_for_status()
result = response.json()

In [ ]:
print(result["exercises"][0])

# It was a sucess, so now the only thing we are gonna use for this is
* Exercise	
* Duration	
* Calories

That is called
* name
* duration_min
* nf_calories

SO lets access to that

In [ ]:
print(result["exercises"][0]['name'])
print(result["exercises"][0]['duration_min'])
print(result["exercises"][0]['nf_calories'])

# SO maybe we wanted to asked the user what excersice he did instead of only putting default data

In [ ]:
answer = input("What excersices you did: ")

#We only will fill for now the required one 
parameters = {
    "query": answer
}

In [ ]:
response = requests.post(url=Full_post_endpoint, headers=header, json=parameters)
response.raise_for_status()
result = response.json()

In [ ]:
print(result["exercises"][0])

# Now we have created an endpoint to use the API, Sheety

## this is the endpoint
> https://api.sheety.co/56df9d1d9fe8908c5a3bcfe5d729cf6b/workoutTracking/workouts

And of course we create an authentication for that on the header

## Lets see an example with **get** operation

In [ ]:
load_dotenv(env_path)
TOKEN_SHEETY = os.getenv('AUTHORIZATION_Sheety')

In [ ]:
#we need a header
header_sheety={
    'Authorization':TOKEN_SHEETY,
}

end_point_sheety = 'https://api.sheety.co/56df9d1d9fe8908c5a3bcfe5d729cf6b/workoutTracking/workouts'

In [ ]:
response = requests.get(url=end_point_sheety, headers=header_sheety)

response.raise_for_status()

data = response.json()

In [ ]:
print(data)

In [ ]:
print(len(data['workouts']))
last_item = len(data['workouts'])-1
#get the last id
id = data['workouts'][last_item]['id']

print(id)

# As we can see we have a dictionary
**workouts**
that contains a list of dictionaries

In [ ]:
print(data['workouts'][0])
print(data['workouts'][1])

# How do we make a **post** service

In [ ]:
#we need the parameters
#but the documentation says that

'''
Sheety expects our record to be nested in a root property
Our ourl ends on workouts, this need to be nested like workout, so
'''

workout_parameters={
    #nested
    'workout':{
    #the required parameters
        'date': '22/07/2022', 
        'time': '4:00:00 PM', 
        'exercise': 'Running', 
        'duration': 40, 
        'calories': 170, 
        'id': 4,
    }
}
#the end point is the same

In [ ]:
response = requests.post(url=end_point_sheety, json=workout_parameters, headers=header_sheety)

print(response.text)

### With this now we can send any data to our sheet

# Final code

In [1]:
#libraries
from pathlib import Path
import requests
from dotenv import load_dotenv
import os
import datetime as dt

In [2]:
#keys, and endpoints
#load our file 
env_path = str(Path.cwd().parent)+'/.env'
load_dotenv(env_path)

#keys
NUTRITION_KEY = os.getenv('Nutrition_APIKey')
APP_ID = os.getenv('App_Nutrition_ID')
TOKEN_SHEETY = os.getenv('AUTHORIZATION_Sheety')

#headers
HEADER_NUTRITION={
    'x-app-id': APP_ID,
    'x-app-key': NUTRITION_KEY,
}
HEADER_SHEETY={
    'Authorization':TOKEN_SHEETY,
}

#endpoints
Base_endpoint = 'https://app.100daysofpython.dev'
Post_endpoint = '/v1/nutrition/natural/exercise'
FULL_NUTRI_POST_ENDPOINT = Base_endpoint+Post_endpoint

ENDPOINT_SHEETY = 'https://api.sheety.co/56df9d1d9fe8908c5a3bcfe5d729cf6b/workoutTracking/workouts'

In [3]:
#functions
def getTime():
    today = dt.datetime.now()
    #format for MM/DD/YYYY
    date=today.strftime('%m/%d/%Y')
    #time as HOURS:MINS:SECONDS
    time = today.strftime('%X')
    return date, time 

In [11]:

def getUserWorkout():
    workourt = input("Tell me, What exercises have you done today?: ")
    parameters = {
        "query": workourt
    }
    return parameters

In [5]:

def dividedAnswer(dataNutry):
    name = (dataNutry["exercises"][0]['name']).title()
    duration = dataNutry["exercises"][0]['duration_min']
    calories = dataNutry["exercises"][0]['nf_calories']
    return name, duration, calories

In [6]:

def checkCurrentID(dataSheety):
    last_item= len(dataSheety['workouts'])-1
    id = dataSheety['workouts'][last_item]['id']
    return id

In [7]:
def prepareDataToSent(dataNutry, dataSheety):
    #get the time
    date, time = getTime()
    #get the data
    name, duration, calories = dividedAnswer(dataNutry)
    #check current id
    id = checkCurrentID(dataSheety)

    workout_parameters={
        'workout':{
        #the required parameters
            'date': date, 
            'time': time, 
            'exercise': name, 
            'duration': duration, 
            'calories': calories, 
            'id': id,
        }
    }
    return workout_parameters 

In [8]:
def sendData(parameters):
    response = requests.post(url=ENDPOINT_SHEETY, json=parameters, headers=HEADER_SHEETY)
    print(response.text)

In [9]:
#rest of code
def main():
    #first we need to asked for the user 
    parameters = getUserWorkout()
    #make a post with the parameters to natty
    dataNutry = (requests.post(url=FULL_NUTRI_POST_ENDPOINT, json=parameters, headers=HEADER_NUTRITION)).json()
    #Then we only need to get the data for Sheety
    datashety = (requests.get(url=ENDPOINT_SHEETY, headers=HEADER_SHEETY)).json()
    all_parameters = prepareDataToSent(dataNutry, datashety)
    #just send the parameters
    sendData(all_parameters)   

In [10]:
main()

{
  "workout": {
    "date": "10/20/2026",
    "time": "10:19:46",
    "exercise": "Running",
    "duration": 60,
    "calories": 720,
    "id": 7
  }
}


# And that means that the data was sent correctly 